# UMSS on MASSVIS

Do our two UMSS stages give the paper's results?

- **S1** predicts three temporal saliency maps (0–0.5 s, 0.5–3 s, 3–5 s). Compared with Table 2.
- **S2** samples scanpaths from those maps. Compared with Table 1.

Each table shows the paper, the authors' released outputs and ours, all scored with the same evaluation
(the authors' code in `official_repository/`). The released outputs check the evaluation: they should give the
paper's numbers. Our maps come from the MD-EAM checkpoint retrained in this repository
(`data/ours/npz/`, exported with `models.umss.export_maps`). S1 scores the chart-size PNGs the export
writes in `png/` with the authors' `predict_and_save_md`; exports without them fall back to the grid maps.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'models/umss').is_dir())
sys.path.insert(0, str(ROOT))

from reproductions.umss_massvis import replication as R

RESULTS = R.EXPERIMENT / 'results'
RESULTS.mkdir(exist_ok=True)

## Data

Wang, Bâce and Bulling, *Scanpath Prediction on Information Visualisations*, TVCG 2023.
The evaluation uses the 66 charts of the published split (every sixth of the 393 MASSVIS charts).

The first time, restore the deposit with `sync_official_repository.py` and run `R.download()` to unzip the
images and released outputs and fetch the MASSVIS fixations and element labels.

In [2]:
# R.download()
all_names, names, paths = R.images()
print('MASSVIS charts:', len(all_names), '| evaluation charts:', len(names))

MASSVIS charts: 393 | evaluation charts: 66


## S1: saliency maps (Table 2)

NSS, CC, KL and SIM of the 0.5–3 s and 3–5 s maps against all human fixations up to 3 s and 5 s
(Gaussian sigma 19 px), as in the authors' evaluation. Higher is better except for KL.

In [3]:
s1 = R.s1()
s1.to_csv(RESULTS / 's1.csv')
display(s1.round(3))

Paper  Released maps   Ours
duration metric                             
3s       NSS     1.406          1.406  1.577
         CC      0.586          0.586  0.663
         KL      0.754          0.754  0.677
         SIM     0.516          0.516  0.543
5s       NSS     1.024          1.024  1.071
         CC      0.514          0.514  0.540
         KL      0.689          0.689  0.687
         SIM     0.530          0.530  0.533

## S2: scanpaths (Table 1)

40 scanpaths per chart from our maps with `UMSS.run(n_fix=None)`, the official UMSS-S2 sampler (run r uses seed r).
They are compared with the viewers' first 5 seconds by Sequence Score, scaled TDE, DTW and MultiMatch.
Higher is better except for DTW. ScanMatch needs MATLAB and is not computed.

This step takes about an hour on 14 cores: the official sampler filters a full-size Gaussian per fixation.

In [ ]:
s2 = R.s2()
s2.to_csv(RESULTS / 's2.csv')
display(s2.round(3))

## Notes

- The paper's 5 s human scanpaths and accumulated human maps are not in the deposit; they are rebuilt from
  MASSVIS with the rules of the authors' code. The released scanpaths do not give back the paper's
  Sequence Score and DTW best, which depend on that rebuild.
- The current checkpoint was retrained with the authors' recipe and its epoch was chosen on these same 66
  charts, so better scores than the paper do not show better generalization. Training now keeps the last
  epoch, as the authors' notebook does.
- S2 is random: another seed gives slightly different numbers.